# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Dataset attached at `/kaggle/input/datasets/wenhaolu49/cipher`.

| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35% | Plan quality vs. oracle beam search |
| Calibration | 25% | Brier score on stated self-knowledge |
| Attention | 20% | Rank correlation on unknown importance |
| Executive | 20% | Plan structure, named risks, alternative plans |

In [ ]:
# 1. Install dependencies
import subprocess, sys

def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

_pip("kaggle-benchmarks", "pandas")
print("Dependencies installed.")

In [ ]:
# 2. Imports
import os, sys, json
import pandas as pd
import kaggle_benchmarks as kbench
from typing import Any, Dict, List, Optional
from kaggle_benchmarks.prompting import ResponseParsingError

# Locate cipher package (Kaggle dataset or local)
_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _c in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_c, "cipher")):
        CIPHER_ROOT = _c
        break
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/ package not found — attach the cipher dataset.")
if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)
print(f"cipher root: {CIPHER_ROOT}")

from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect
from cipher.schema import validate_response
from cipher.scorer import score_response
print("cipher imports OK.")

In [ ]:
# 3. Load instances
DATA_PATH = next(
    (p for p in [
        os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
        os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
    ] if os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError("data/instances.jsonl not found.")

# 10 for quick dev runs (~10 min), 50 for testing (~55 min), None for full 1000-instance submission
KBENCH_LIMIT = 10

with open(DATA_PATH) as f:
    ALL_RECORDS = [json.loads(line) for line in f if line.strip()]
if KBENCH_LIMIT:
    ALL_RECORDS = ALL_RECORDS[:KBENCH_LIMIT]

print(f"Loaded {len(ALL_RECORDS)} instances from {DATA_PATH}")
print("\nAvailable models:", list(kbench.llms.keys()))

In [ ]:
# 4. Scoring helpers
def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    h   = rec["hidden"]
    hrl = {e["rule_name"]: set(e["hidden"]) for e in h.get("hidden_fields", [])}
    rules = []
    for r in h["rules"]:
        hides = hrl.get(r["name"], set())
        t, e  = r["trigger"], r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(kind=t["kind"], i=t["i"], j=t.get("j",-1), k=t.get("k",0),
                            hidden_kind=("trigger_kind" in hides), hidden_k=("trigger_k" in hides)),
            effect=Effect(kind=e["kind"], target=e["target"],
                          delta=e.get("delta",0), source=e.get("source",-1),
                          hidden_kind=("effect_kind" in hides), hidden_delta=("effect_delta" in hides)),
        ))
    initial = State(tuple(EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        public_rule_descriptions=[], hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )

_ZERO = dict(composite=0.0, objective=0.0, calibration=0.0, attention=0.0, executive=0.0)

def _score(raw_text: str, rec: Dict[str, Any]) -> Dict[str, float]:
    start, end = raw_text.find("{"), raw_text.rfind("}")
    if start == -1:
        return _ZERO.copy()
    try:
        raw  = json.loads(raw_text[start:end+1])
        inst = _instance_from_record(rec)
        resp = validate_response(raw)
        bd   = score_response(resp, inst,
                              best_obj=rec["hidden"].get("oracle_best"),
                              worst_obj=rec["hidden"].get("oracle_worst"))
        return bd.to_dict()
    except Exception:
        return _ZERO.copy()

In [ ]:
# 5. Per-instance task — returns all four dimensions
@kbench.task(name="cipher_single_instance_scorer", store_task=False)
def cipher_single_instance_scorer(llm, prompt: str, record_json: str) -> dict:
    try:
        reply = llm.prompt(prompt)
        rec   = json.loads(record_json)
        return _score(str(reply), rec)
    except (ResponseParsingError, json.JSONDecodeError, Exception):
        return _ZERO.copy()

print("cipher_single_instance_scorer defined.")

In [ ]:
# 6. Batch task — aggregates all dimensions and returns composite score
@kbench.task(name="cipher_metacognition_evaluation")
def cipher_metacognition_evaluation(llm):
    """CIPHER: micro-worlds with hidden causal rules in invented vocabulary. Tests metacognitive calibration."""
    evaluation_df = pd.DataFrame([{
        "prompt":      rec["prompt"],
        "record_json": json.dumps(rec),
    } for rec in ALL_RECORDS])

    with kbench.client.enable_cache():
        runs = cipher_single_instance_scorer.evaluate(
            llm=[llm],
            evaluation_data=evaluation_df,
            n_jobs=2,
            remove_run_files=True,
        )

    results_df = runs.as_dataframe()

    def _mean(key):
        vals = [r.get(key, 0.0) for r in results_df["result"] if isinstance(r, dict)]
        return float(sum(vals) / max(len(vals), 1))

    scores = {
        "composite":   _mean("composite"),
        "objective":   _mean("objective"),
        "calibration": _mean("calibration"),
        "attention":   _mean("attention"),
        "executive":   _mean("executive"),
    }

    print(f"composite={scores['composite']:.3f}  obj={scores['objective']:.3f}  "
          f"cal={scores['calibration']:.3f}  att={scores['attention']:.3f}  exe={scores['executive']:.3f}")
    return scores["composite"]

cipher_metacognition_evaluation.run(kbench.llm)

In [ ]:
%choose cipher_metacognition_evaluation